# 00 · Cuaderno de verificación

Si este cuaderno corre de principio a fin sin errores, el entorno está
correctamente levantado. Este es el criterio de aceptación de T2:
otra persona clona el repositorio, corre `docker compose up`, abre este
cuaderno y lo ejecuta completo sin tocar nada más.

In [1]:
import pandas as pd
import numpy as np
import sys

print(f"Python: {sys.version}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

Python: 3.13.14 | packaged by conda-forge | (main, Jun 12 2026, 09:50:25) [GCC 14.3.0]
pandas: 2.2.3
numpy: 2.1.1


In [2]:
import os

# Verifica que las rutas montadas por Docker Compose son visibles
# (subimos un nivel con ../ porque el notebook vive en notebooks/)
for carpeta in ["../data/raw", "../src", "../docs"]:
    existe = os.path.isdir(carpeta)
    print(f"{carpeta}: {'OK' if existe else 'NO ENCONTRADA'}")

../data/raw: OK
../src: OK
../docs: OK


In [3]:
# Prueba minima con pandas para confirmar que el entorno ejecuta codigo real
df_prueba = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
print(df_prueba)
print()
print("Si ves la tabla de arriba sin errores, el entorno reproducible funciona.")

   a  b
0  1  4
1  2  5
2  3  6

Si ves la tabla de arriba sin errores, el entorno reproducible funciona.


## 5 · Verificación de conexión a PostgreSQL (segundo servicio)

Confirma que el contenedor de Jupyter puede alcanzar el contenedor de
PostgreSQL por la red interna de Docker Compose, usando el nombre del
servicio (`db`) en vez de `localhost` y credenciales tomadas de
variables de entorno (nunca escritas en el código).

In [4]:
import os
from sqlalchemy import create_engine, text

usuario = os.environ["POSTGRES_USER"]
clave = os.environ["POSTGRES_PASSWORD"]
host = os.environ["POSTGRES_HOST"]  # "db", el nombre del servicio, no localhost
puerto = os.environ["POSTGRES_PORT"]
base = os.environ["POSTGRES_DB"]

url = f"postgresql+psycopg2://{usuario}:{clave}@{host}:{puerto}/{base}"
engine = create_engine(url)

with engine.connect() as conexion:
    resultado = conexion.execute(text("SELECT version();"))
    print("Conexion exitosa a PostgreSQL:")
    print(resultado.fetchone()[0])

Conexion exitosa a PostgreSQL:
PostgreSQL 16.4 (Debian 16.4-1.pgdg120+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14) 12.2.0, 64-bit
